# EDSS 공식 학교명–OpenID 교차표 감사

## TL;DR

2026년 8월 공식 EDSS 제공목록과 체크섬이 검증된 2023·2024 취업통계 원본을 대조한다. 공식 교차표가 없으면 구조적 후보를 정식 OpenID로 승격하지 않는다.

## Context & Methods

- 공식 제공목록의 취업통계 2023–2024 행에서 학교코드 제공 여부와 제공항목을 확인한다.
- 실제 중첩 ZIP을 풀어 내부 CSV 헤더를 직접 읽고 학교명·개방ID·코드형 열을 검사한다.
- 파일 SHA-256을 저장된 감사 결과와 대조한다.
- 공식 사이트의 `개방ID`, `학교코드` 검색 결과 수를 확인한다.
- 후보 규칙과 공식 확정을 구분하고 정식 ID 대입 건수를 검산한다.

## Data

In [1]:
from pathlib import Path
import json, sys
from IPython.display import Markdown, display

ROOT = Path.cwd()
if not (ROOT / 'data/metadata').exists():
    ROOT = ROOT.parent
assert (ROOT / 'data/metadata').exists(), 'repository root not found'
sys.path.insert(0, str(ROOT / 'scripts'))
from audit_edss_official_crosswalk import (
    nested_csv_header, provider_employment_evidence, sha256_file
)
audit_path = ROOT / 'data/metadata/edss_official_crosswalk_audit.json'
audit = json.loads(audit_path.read_text(encoding='utf-8'))
display(Markdown(f"감사 버전 {audit['audit_version']}, 원본 ZIP {len(audit['employment_raw_headers'])}개를 읽었다."))

감사 버전 1, 원본 ZIP 2개를 읽었다.

## Results

In [2]:
provider = audit['official_provider_list']
provider_path = ROOT / provider['path']
assert provider_path.exists()
assert sha256_file(provider_path) == provider['sha256']
recomputed_provider = provider_employment_evidence(provider_path)
assert recomputed_provider['listed_school_code_provided'] == 'Y'
assert recomputed_provider['listed_has_school_name']
assert not recomputed_provider['listed_has_open_id']

raw_rows = []
for recorded in audit['employment_raw_headers']:
    path = ROOT / recorded['path']
    assert sha256_file(path) == recorded['sha256']
    current = nested_csv_header(path)
    assert current['column_count'] == 24
    assert current['has_school_name']
    assert not current['has_open_id']
    assert current['school_code_fields'] == []
    raw_rows.append((recorded['year'], current['column_count'], '있음', '없음', '없음'))

table = '| 연도 | 열 수 | 학교명 | 개방ID | 기타 학교코드 |\n|---|---:|---|---|---|\n'
table += '\n'.join(f'| {year} | {cols} | {name} | {oid} | {code} |' for year, cols, name, oid, code in raw_rows)
display(Markdown(table))

| 연도 | 열 수 | 학교명 | 개방ID | 기타 학교코드 |
|---|---:|---|---|---|
| 2023 | 24 | 있음 | 없음 | 없음 |
| 2024 | 24 | 있음 | 없음 | 없음 |

In [3]:
conclusion = audit['crosswalk_conclusion']
candidate = audit['candidate_resolution']
assert conclusion['metadata_conflict']
assert not conclusion['official_crosswalk_available']
assert candidate['context_confirmed_candidate_identity_count'] == 30
assert candidate['officially_confirmed_candidate_identity_count'] == 0
assert candidate['canonical_open_id_imputed_row_count'] == 0
assert [item['result_count'] for item in audit['official_site_search']] == [0, 0]
rows = [
    ('구조·0101 맥락 일치 후보', '30'),
    ('공식 교차표로 확정된 후보', '0'),
    ('정식 개방ID 대입', '0'),
    ('공식 목록–실제 파일 충돌', '있음'),
]
table = '| 판정 | 결과 |\n|---|---:|\n' + '\n'.join(f'| {k} | {v} |' for k, v in rows)
display(Markdown(table))

| 판정 | 결과 |
|---|---:|
| 구조·0101 맥락 일치 후보 | 30 |
| 공식 교차표로 확정된 후보 | 0 |
| 정식 개방ID 대입 | 0 |
| 공식 목록–실제 파일 충돌 | 있음 |

## Takeaways

- 공식 EDSS 제공목록은 2023–2024 취업통계의 학교코드 제공 여부를 `Y`로 표시한다.
- 실제 두 원본 파일은 각각 24열이며 `학교명`은 있지만 `개방ID`와 다른 코드형 학교 열은 없다.
- 공식 사이트에서 `개방ID`와 `학교코드` 검색 결과도 각각 0건이다.
- 따라서 후보 30개는 구조적으로 강한 증거일 뿐 공식 매핑이 아니며, EDSS의 교차표나 서면 설명 전에는 확정하지 않는다.